# Final Deployment Package Audit

Ce notebook reprend le script `final_deployment_package_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Audit de readiness du package de scoring live et des artefacts runtime.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Audit deployment readiness for final machine-safety scoring artifacts.
- Commande de reproduction referencee : final deployment package audit.
- Artefacts controles : Final deployment/scoring package audit exists. (`runs/exp_065_final_deployment_package_with_pose_backend/metrics/deployment_readiness_checklist.csv`).
- Run par defaut : `runs/exp_048_final_deployment_package`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_deployment_package_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd

from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, make_run_dir


FINAL_SCORE_COLS = [
    "final_sequence_only",
    "final_attention_rule",
    "final_ppe_rule",
    "final_full_rule",
    "final_safety_sensitive_rule",
    "final_high_sensitivity_rule",
    "final_attention_ppe_prior",
    "final_learned_meta_mean",
]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `read_json`

Cette cellule definit `read_json`. Elle prepare une partie du script.

In [ ]:
def read_json(path):
    path = resolve(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


## Fonction `file_info`

Cette cellule definit `file_info`. Elle prepare une partie du script.

In [ ]:
def file_info(path):
    path = resolve(path)
    return {
        "path": str(path),
        "exists": bool(path.exists()),
        "bytes": int(path.stat().st_size) if path.exists() else 0,
    }


## Fonction `glob_infos`

Cette cellule definit `glob_infos`. Elle prepare une partie du script.

In [ ]:
def glob_infos(base, pattern):
    base = resolve(base)
    rows = []
    for path in sorted(base.glob(pattern)):
        rows.append(file_info(path))
    return rows


## Fonction `finite_01`

Cette cellule definit `finite_01`. Elle prepare une partie du script.

In [ ]:
def finite_01(series):
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64)
    finite = np.isfinite(values)
    return {
        "finite": bool(finite.all()),
        "min": float(np.nanmin(values)) if len(values) else np.nan,
        "max": float(np.nanmax(values)) if len(values) else np.nan,
        "within_0_1": bool(finite.all() and np.nanmin(values) >= -1e-6 and np.nanmax(values) <= 1.0 + 1e-6) if len(values) else False,
    }


## Fonction `validate_final_score_contract`

Cette cellule definit `validate_final_score_contract`. Elle prepare une partie du script.

In [ ]:
def validate_final_score_contract(final_run):
    final_run = resolve(final_run)
    rows = []
    for path in sorted((final_run / "features").glob("final_scores_seed*.csv")):
        seed = path.stem.replace("final_scores_seed", "")
        df = pd.read_csv(path)
        sequence_cols = [col for col in df.columns if col.startswith("sequence_") and col not in {"sequence_mean_tcn", "sequence_max_tcn"}]
        learned_cols = [col for col in df.columns if col.startswith("learned_meta_") and col != "learned_meta_mean"]
        checks = {}
        if sequence_cols:
            checks["sequence_mean_tcn"] = df[sequence_cols].mean(axis=1)
            checks["sequence_max_tcn"] = df[sequence_cols].max(axis=1)
        if learned_cols:
            checks["learned_meta_mean"] = df[learned_cols].mean(axis=1)
            checks["final_learned_meta_mean"] = checks["learned_meta_mean"]
        d = df["sequence_mean_tcn"].clip(0, 1)
        a = df["attention_risk"].clip(0, 1)
        p = df["ppe_risk"].clip(0, 1)
        checks["final_sequence_only"] = d
        checks["final_attention_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.30 * a)
        checks["final_ppe_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.25 * p)
        checks["final_full_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.30 * a) * (1.0 - 0.25 * p)
        checks["final_safety_sensitive_rule"] = 1.0 - (1.0 - d) * (1.0 - 0.45 * a) * (1.0 - 0.35 * p)
        checks["final_high_sensitivity_rule"] = np.clip(d + (1.0 - d) * (0.40 * a + 0.30 * p + 0.20 * a * p), 0.0, 1.0)
        checks["final_attention_ppe_prior"] = np.maximum(d, np.clip(0.25 * a + 0.20 * p + 0.10 * a * p, 0.0, 1.0))

        for col, expected in checks.items():
            if col not in df.columns:
                rows.append(
                    {
                        "seed": seed,
                        "check": col,
                        "exists": False,
                        "max_abs_error": np.nan,
                        "finite": False,
                        "within_0_1": False,
                        "pass": False,
                    }
                )
                continue
            err = np.max(np.abs(pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64) - np.asarray(expected, dtype=np.float64)))
            bounds = finite_01(df[col])
            rows.append(
                {
                    "seed": seed,
                    "check": col,
                    "exists": True,
                    "max_abs_error": float(err),
                    "finite": bounds["finite"],
                    "within_0_1": bounds["within_0_1"],
                    "pass": bool(err <= 1e-6 and bounds["finite"] and bounds["within_0_1"]),
                }
            )
        for col in ["attention_risk", "ppe_risk", *FINAL_SCORE_COLS]:
            if col in df.columns:
                bounds = finite_01(df[col])
                rows.append(
                    {
                        "seed": seed,
                        "check": f"{col}_range",
                        "exists": True,
                        "max_abs_error": 0.0,
                        "finite": bounds["finite"],
                        "within_0_1": bounds["within_0_1"],
                        "pass": bool(bounds["finite"] and bounds["within_0_1"]),
                    }
                )
    return pd.DataFrame(rows)


## Fonction `artifact_group_summary`

Cette cellule definit `artifact_group_summary`. Elle prepare une partie du script.

In [ ]:
def artifact_group_summary(args):
    fusion = resolve(args.fusion_run)
    final = resolve(args.final_score_run)
    rows = []
    groups = [
        ("pose_detector_pt", [ROOT / "yolo11n-pose.pt"], 1, "required for raw-video YOLO pose extraction"),
        ("pose_detector_onnx", [ROOT / "yolo11n-pose.onnx"], 1, "optional ONNX pose extraction artifact"),
        ("single_sequence_inference_script", [ROOT / "sequence_inference_demo.py"], 1, "existing single TCN inference script"),
        ("single_sequence_model", [resolve(args.sequence_model)], 1, "raw-video smoke-tested danger sequence checkpoint"),
        ("single_sequence_features", [resolve(args.sequence_features)], 1, "normalizer and sequence metadata source for single-model inference"),
        ("final_sequence_seed_models", sorted((fusion / "models").glob("seed*_tcn*.pt")), 9, "same-split TCN seed checkpoints used for final score research artifacts"),
        ("final_crop_models", sorted((fusion / "models").glob("attention_seed*.pt")) + sorted((fusion / "models").glob("blouse_seed*.pt")), 12, "attention and blouse/PPE crop checkpoints trained on shared parent splits"),
        ("final_meta_models", sorted((fusion / "models").glob("meta_seed*.joblib")), 9, "learned meta fusion models"),
        ("final_sequence_normalizers", sorted((fusion / "features").glob("sequence_normalizer_seed*.npz")), 3, "per-seed train-only sequence normalizers"),
        ("final_fused_prediction_tables", sorted((fusion / "features").glob("same_split_fused_seed*.csv")), 9, "offline fused prediction tables"),
        ("final_score_tables", sorted((final / "features").glob("final_scores_seed*.csv")), 3, "final aggregated score tables"),
        ("final_score_metrics", [final / "metrics" / "final_score_operating_summary.csv"], 1, "window-level final score metrics"),
        ("final_subclip_metrics", [resolve(args.final_subclip_run) / "metrics" / "final_score_subclip_validation_selected_summary.csv"], 1, "subclip-level final score metrics"),
        ("precomputed_pose_smoke", [resolve(args.precomputed_smoke) / "inference_config.json", resolve(args.precomputed_smoke) / "risk_predictions.csv", resolve(args.precomputed_smoke) / "top_risk_contact_sheet.jpg"], 3, "single-sequence precomputed-pose inference smoke"),
        ("raw_video_smoke", [resolve(args.raw_smoke) / "inference_config.json", resolve(args.raw_smoke) / "risk_predictions.csv", resolve(args.raw_smoke) / "top_risk_contact_sheet.jpg"], 3, "single-sequence raw-video inference smoke"),
        ("final_fused_raw_video_smoke", [resolve(args.final_fused_smoke) / "final_fused_inference_config.json", resolve(args.final_fused_smoke) / "final_fused_risk_predictions.csv", resolve(args.final_fused_smoke) / "top_final_fused_risk_contact_sheet.jpg", resolve(args.final_fused_smoke) / "crop_risk_summary.csv"], 4, "full final trajectory+attention+PPE raw-video smoke"),
        ("causal_final_fused_stream_smoke", [resolve(args.causal_fused_smoke) / "causal_fused_stream_config.json", resolve(args.causal_fused_smoke) / "causal_fused_stream_predictions.csv", resolve(args.causal_fused_smoke) / "top_final_fused_risk_contact_sheet.jpg", resolve(args.causal_fused_smoke) / "causal_latency_summary.csv"], 4, "causal frame-by-frame final trajectory+attention+PPE raw-video smoke with rolling crop risk"),
        ("realtime_variant_benchmark", [resolve(args.runtime_variant_run) / "runtime_variant_benchmark_config.json", resolve(args.runtime_variant_run) / "runtime_variant_predictions.csv", resolve(args.runtime_variant_run) / "runtime_variant_total_summary.csv", resolve(args.runtime_variant_run) / "optimized_reference_comparison.csv"], 4, "optimized causal runtime benchmark for full, lightweight fused, and sequence-only deployment variants"),
        ("runtime_resolution_tradeoff", [resolve(args.resolution_tradeoff_run) / "runtime_resolution_tradeoff_summary.md", resolve(args.resolution_tradeoff_run) / "metrics" / "runtime_resolution_tradeoff.csv"], 2, "pose-resolution runtime and score-stability tradeoff audit"),
        ("runtime_pose_backend_audit", [resolve(args.pose_backend_run) / "runtime_pose_backend_audit_summary.md", resolve(args.pose_backend_run) / "runtime_pose_backend_audit.csv"], 2, "PyTorch-vs-ONNX pose backend runtime and score-stability audit"),
    ]
    for name, paths, expected, role in groups:
        infos = [file_info(path) for path in paths]
        existing = [item for item in infos if item["exists"] and item["bytes"] > 0]
        rows.append(
            {
                "group": name,
                "role": role,
                "expected_count": int(expected),
                "found_count": int(len(existing)),
                "bytes_total": int(sum(item["bytes"] for item in existing)),
                "pass": bool(len(existing) >= expected),
                "paths": "|".join(item["path"] for item in infos),
            }
        )
    return pd.DataFrame(rows)


## Fonction `loadability_smoke`

Cette cellule definit `loadability_smoke`. Elle prepare une partie du script.

In [ ]:
def loadability_smoke(args):
    rows = []
    try:
        import torch

        path = resolve(args.sequence_model)
        payload = torch.load(path, map_location="cpu", weights_only=False)
        rows.append(
            {
                "artifact": "single_sequence_model",
                "path": str(path),
                "loadable": True,
                "details": f"kind={payload.get('kind')} seq_len={payload.get('seq_len')} horizons={payload.get('horizons')}",
            }
        )
    except Exception as exc:
        rows.append({"artifact": "single_sequence_model", "path": str(resolve(args.sequence_model)), "loadable": False, "details": repr(exc)})

    try:
        import joblib

        meta_paths = sorted((resolve(args.fusion_run) / "models").glob("meta_seed*.joblib"))
        if not meta_paths:
            raise FileNotFoundError("no meta_seed*.joblib files")
        payload = joblib.load(meta_paths[0])
        rows.append(
            {
                "artifact": "sample_meta_model",
                "path": str(meta_paths[0]),
                "loadable": True,
                "details": f"keys={sorted(payload.keys())}",
            }
        )
    except Exception as exc:
        rows.append({"artifact": "sample_meta_model", "path": str(resolve(args.fusion_run) / "models"), "loadable": False, "details": repr(exc)})

    try:
        import torch

        crop_paths = sorted((resolve(args.fusion_run) / "models").glob("attention_seed*.pt"))
        if not crop_paths:
            raise FileNotFoundError("no attention_seed*.pt files")
        payload = torch.load(crop_paths[0], map_location="cpu", weights_only=False)
        keys = sorted(payload.keys()) if isinstance(payload, dict) else [type(payload).__name__]
        rows.append(
            {
                "artifact": "sample_crop_model",
                "path": str(crop_paths[0]),
                "loadable": True,
                "details": f"payload_keys={keys[:8]}",
            }
        )
    except Exception as exc:
        rows.append({"artifact": "sample_crop_model", "path": str(resolve(args.fusion_run) / "models"), "loadable": False, "details": repr(exc)})
    return pd.DataFrame(rows)


## Fonction `inference_smoke_rows`

Cette cellule definit `inference_smoke_rows`. Elle prepare une partie du script.

In [ ]:
def inference_smoke_rows(args):
    rows = []
    for name, path in [("precomputed_pose_single_tcn", args.precomputed_smoke), ("raw_video_single_tcn", args.raw_smoke)]:
        run_dir = resolve(path)
        config = read_json(run_dir / "inference_config.json")
        pred_path = run_dir / "risk_predictions.csv"
        pred = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
        rows.append(
            {
                "smoke": name,
                "run_dir": str(run_dir),
                "config_exists": bool(config),
                "prediction_rows": int(len(pred)),
                "max_risk": float(config.get("max_risk", np.nan)) if config else np.nan,
                "first_alarm_time_s": config.get("first_alarm_time_s") if config else None,
                "contact_sheet_exists": bool((run_dir / "top_risk_contact_sheet.jpg").exists()),
                "mode": "raw_video" if config.get("raw_video_mode") else "precomputed_pose",
                "pass": bool(config and len(pred) > 0 and (run_dir / "top_risk_contact_sheet.jpg").exists()),
            }
        )
    run_dir = resolve(args.final_fused_smoke)
    config = read_json(run_dir / "final_fused_inference_config.json")
    pred_path = run_dir / "final_fused_risk_predictions.csv"
    pred = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
    rows.append(
        {
            "smoke": "raw_video_final_fused",
            "run_dir": str(run_dir),
            "config_exists": bool(config),
            "prediction_rows": int(len(pred)),
            "max_risk": float(config.get("max_score", np.nan)) if config else np.nan,
            "first_alarm_time_s": config.get("first_alarm_time_s") if config else None,
            "contact_sheet_exists": bool((run_dir / "top_final_fused_risk_contact_sheet.jpg").exists()),
            "mode": "raw_video_final_fused",
            "pass": bool(config and len(pred) > 0 and (run_dir / "top_final_fused_risk_contact_sheet.jpg").exists()),
        }
    )
    run_dir = resolve(args.causal_fused_smoke)
    config = read_json(run_dir / "causal_fused_stream_config.json")
    pred_path = run_dir / "causal_fused_stream_predictions.csv"
    pred = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
    rows.append(
        {
            "smoke": "causal_raw_video_final_fused",
            "run_dir": str(run_dir),
            "config_exists": bool(config),
            "prediction_rows": int(len(pred)),
            "max_risk": float(config.get("max_score", np.nan)) if config else np.nan,
            "first_alarm_time_s": config.get("first_alarm_time_s") if config else None,
            "contact_sheet_exists": bool((run_dir / "top_final_fused_risk_contact_sheet.jpg").exists()),
            "mode": "causal_raw_video_final_fused",
            "mean_latency_ms": float(config.get("mean_total_latency_ms", np.nan)) if config else np.nan,
            "estimated_fps": float(config.get("estimated_fps_from_mean_total_latency", np.nan)) if config else np.nan,
            "pass": bool(config and len(pred) > 0 and (run_dir / "top_final_fused_risk_contact_sheet.jpg").exists()),
        }
    )
    return pd.DataFrame(rows)


## Fonction `runtime_variant_summary`

Cette cellule definit `runtime_variant_summary`. Elle prepare une partie du script.

In [ ]:
def runtime_variant_summary(args):
    path = resolve(args.runtime_variant_run) / "runtime_variant_total_summary.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


## Fonction `runtime_resolution_summary`

Cette cellule definit `runtime_resolution_summary`. Elle prepare une partie du script.

In [ ]:
def runtime_resolution_summary(args):
    path = resolve(args.resolution_tradeoff_run) / "metrics" / "runtime_resolution_tradeoff.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


## Fonction `runtime_pose_backend_summary`

Cette cellule definit `runtime_pose_backend_summary`. Elle prepare une partie du script.

In [ ]:
def runtime_pose_backend_summary(args):
    path = resolve(args.pose_backend_run) / "runtime_pose_backend_audit.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


## Fonction `readiness_rows`

Cette cellule definit `readiness_rows`. Elle prepare une partie du script.

In [ ]:
def readiness_rows(artifact_summary, contract_df, load_df, smoke_df, runtime_df, resolution_df, pose_backend_df):
    artifact_pass = bool(artifact_summary["pass"].all()) if not artifact_summary.empty else False
    contract_pass = bool(contract_df["pass"].all()) if not contract_df.empty else False
    load_pass = bool(load_df["loadable"].all()) if not load_df.empty else False
    single_sequence_smokes = smoke_df[smoke_df["smoke"].isin(["precomputed_pose_single_tcn", "raw_video_single_tcn"])]
    single_sequence_pass = bool(len(single_sequence_smokes) >= 2 and single_sequence_smokes["pass"].all())
    final_fused_smoke = smoke_df[smoke_df["smoke"].eq("raw_video_final_fused")]
    final_fused_pass = bool(not final_fused_smoke.empty and final_fused_smoke["pass"].all())
    causal_fused_smoke = smoke_df[smoke_df["smoke"].eq("causal_raw_video_final_fused")]
    causal_fused_pass = bool(not causal_fused_smoke.empty and causal_fused_smoke["pass"].all())
    causal_fps = float(causal_fused_smoke["estimated_fps"].iloc[0]) if causal_fused_pass and "estimated_fps" in causal_fused_smoke else np.nan
    runtime_present = not runtime_df.empty
    sequence_runtime = runtime_df[runtime_df["variant"].eq("sequence_only_seed111_tcn_focal")] if runtime_present else pd.DataFrame()
    core_sequence_fps = float(sequence_runtime["estimated_fps_from_steady_mean"].iloc[0]) if not sequence_runtime.empty else np.nan
    fused_runtime = runtime_df[
        runtime_df["variant"].astype(str).ne("sequence_only_seed111_tcn_focal")
    ].copy() if runtime_present else pd.DataFrame()
    if not fused_runtime.empty:
        best_fused = fused_runtime.sort_values("steady_mean_ms").iloc[0]
        best_fused_variant = str(best_fused["variant"])
        best_fused_fps = float(best_fused["estimated_fps_from_steady_mean"])
        best_fused_p95 = float(best_fused["steady_p95_ms"])
    else:
        best_fused_variant = ""
        best_fused_fps = np.nan
        best_fused_p95 = np.nan
    resolution_present = not resolution_df.empty
    best_resolution_fused_variant = ""
    best_resolution_fused_fps = np.nan
    best_resolution_fused_p95 = np.nan
    best_resolution_fused_score_p95 = np.nan
    if resolution_present:
        resolution_fused = resolution_df[
            resolution_df["variant"].astype(str).ne("sequence_only_seed111_tcn_focal")
            & resolution_df["variant"].astype(str).ne("research_full_3seed_learned")
            & resolution_df["variant"].astype(str).ne("single_seed111_learned")
        ].copy()
        if not resolution_fused.empty:
            best_resolution = resolution_fused.sort_values(["pass_mean_30fps", "estimated_fps_from_steady_mean"], ascending=[False, False]).iloc[0]
            best_resolution_fused_variant = f"{best_resolution['variant']}@imgsz{int(best_resolution['imgsz'])}"
            best_resolution_fused_fps = float(best_resolution["estimated_fps_from_steady_mean"])
            best_resolution_fused_p95 = float(best_resolution["steady_p95_ms"])
            best_resolution_fused_score_p95 = float(best_resolution["score_p95_abs_diff_vs_640"])
    production_realtime_pass = bool(
        np.isfinite(best_resolution_fused_fps)
        and best_resolution_fused_fps >= 30.0
        and np.isfinite(best_resolution_fused_p95)
        and best_resolution_fused_p95 <= 33.4
        and np.isfinite(best_resolution_fused_score_p95)
        and best_resolution_fused_score_p95 <= 0.05
    )
    pose_backend_present = not pose_backend_df.empty
    if pose_backend_present:
        pose_backend_best = pose_backend_df.sort_values("estimated_hybrid_cuda_score_total_mean_ms").iloc[0]
        pose_backend_notes = (
            f"ONNX-CPU pose backend audit exists. Best optimistic hybrid is `{pose_backend_best['variant']}` at "
            f"{float(pose_backend_best['estimated_hybrid_cuda_score_fps']):.2f} FPS, "
            f"mean {float(pose_backend_best['estimated_hybrid_cuda_score_total_mean_ms']):.2f} ms, "
            f"p95 {float(pose_backend_best['estimated_hybrid_cuda_score_total_p95_ms']):.2f} ms, "
            f"score-p95-diff {float(pose_backend_best['score_p95_abs_diff']):.3f}."
        )
    else:
        pose_backend_notes = "No pose backend runtime audit found."
    rows = [
        {
            "check": "research_artifacts_present",
            "status": "pass" if artifact_pass else "fail",
            "evidence": "artifact_manifest.csv",
            "notes": "Required checkpoints, prediction tables, metrics, and smoke-test files are present.",
        },
        {
            "check": "final_score_formula_reproducible",
            "status": "pass" if contract_pass else "fail",
            "evidence": "final_score_contract_validation.csv",
            "notes": "Final score CSV columns reproduce the documented formulas within tolerance and stay in [0,1].",
        },
        {
            "check": "sample_model_artifacts_loadable",
            "status": "pass" if load_pass else "fail",
            "evidence": "loadability_smoke.csv",
            "notes": "Representative PyTorch and joblib artifacts load on CPU.",
        },
        {
            "check": "single_sequence_raw_video_runtime",
            "status": "pass" if single_sequence_pass else "fail",
            "evidence": "inference_smoke_audit.csv",
            "notes": "Existing smoke tests run a single TCN danger model from precomputed pose and raw MP4.",
        },
        {
            "check": "full_final_fused_raw_video_runtime",
            "status": "pass" if final_fused_pass else "gap",
            "evidence": "final_fused_inference_demo.py; inference_smoke_audit.csv",
            "notes": "A raw-video smoke wrapper now runs pose, crop attention/PPE, seed sequence models, meta/rule fusion, thresholding, and contact-sheet overlays for the final score.",
        },
        {
            "check": "causal_final_fused_raw_video_runtime",
            "status": "pass" if causal_fused_pass else "gap",
            "evidence": "final_causal_fused_stream_smoke.py; inference_smoke_audit.csv",
            "notes": "A causal raw-video smoke processes frames in order, uses current-frame crop risks with EMA, uses only pose history up to frame t, and records latency.",
        },
        {
            "check": "optimized_runtime_variant_benchmark",
            "status": "pass" if runtime_present else "gap",
            "evidence": "runtime_variant_total_summary.csv",
            "notes": "Optimized causal benchmarks compare the full 3-seed fused stack, single-seed fused stack, fast transparent fused stack, slow-context fused stack, and sequence-only stack.",
        },
        {
            "check": "pose_resolution_runtime_tradeoff",
            "status": "pass" if resolution_present else "gap",
            "evidence": "runtime_resolution_tradeoff.csv",
            "notes": "Pose-resolution sweep tests whether lower YOLO imgsz improves runtime without destabilizing the 640-pixel risk trace.",
        },
        {
            "check": "pose_backend_runtime_audit",
            "status": "pass" if pose_backend_present else "gap",
            "evidence": "runtime_pose_backend_audit.csv",
            "notes": pose_backend_notes,
        },
        {
            "check": "core_sequence_realtime_candidate",
            "status": "pass" if np.isfinite(core_sequence_fps) and core_sequence_fps >= 30.0 else "gap",
            "evidence": "runtime_variant_total_summary.csv",
            "notes": f"The optimized sequence-only seed111 TCN focal candidate reaches {core_sequence_fps:.2f} FPS steady-state mean including YOLO pose; this is a core danger runtime, not the final fused score.",
        },
        {
            "check": "production_realtime_final_fused_runtime",
            "status": "pass" if production_realtime_pass else "gap",
            "evidence": "runtime_variant_total_summary.csv; runtime_resolution_tradeoff.csv; runtime_pose_backend_audit.csv",
            "notes": f"The best fused speed candidate remains `{best_resolution_fused_variant}` at {best_resolution_fused_fps:.2f} FPS mean, p95 {best_resolution_fused_p95:.1f} ms, score-p95-diff {best_resolution_fused_score_p95:.3f} vs 640. ONNX-CPU pose backend was also tested and did not close the gap. It does not satisfy the combined mean-FPS, p95-latency, and score-stability gate.",
        },
    ]
    return pd.DataFrame(rows)


## Fonction `write_csv`

Cette cellule definit `write_csv`. Elle prepare une partie du script.

In [ ]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


## Fonction `write_scoring_contract`

Cette cellule definit `write_scoring_contract`. Elle prepare une partie du script.

In [ ]:
def write_scoring_contract(run_dir):
    lines = ["# Final Scoring Contract", ""]
    lines.append("Inputs per time window:")
    lines.append("")
    lines.append("- `d`: trajectory danger risk from the same-split TCN sequence ensemble.")
    lines.append("- `a`: attention risk, where higher means distracted.")
    lines.append("- `p`: blouse/PPE risk, where higher means badly worn.")
    lines.append("")
    lines.append("All risks are clipped to `[0, 1]` before fusion.")
    lines.append("")
    lines.append("Fusion semantics:")
    lines.append("")
    lines.append("- `d`, `a`, and `p` are model-estimated component risks.")
    lines.append("- The transparent rule weights are human-specified safety policy parameters, not causal weights learned from the current small staged dataset.")
    lines.append("- The intended policy is conservative: distraction and bad PPE lower the allowed tolerance for approaching the machine before physical danger-zone entry.")
    lines.append("- Learned/meta fusion is kept as an ablation and calibration reference, not as the only acceptable deployment score.")
    lines.append("")
    lines.append("Formulas:")
    lines.append("")
    lines.append("```text")
    lines.append("final_sequence_only          = d")
    lines.append("final_attention_rule         = 1 - (1 - d) * (1 - 0.30 * a)")
    lines.append("final_ppe_rule               = 1 - (1 - d) * (1 - 0.25 * p)")
    lines.append("final_full_rule              = 1 - (1 - d) * (1 - 0.30 * a) * (1 - 0.25 * p)")
    lines.append("final_safety_sensitive_rule  = 1 - (1 - d) * (1 - 0.45 * a) * (1 - 0.35 * p)")
    lines.append("final_high_sensitivity_rule  = clip(d + (1 - d) * (0.40*a + 0.30*p + 0.20*a*p), 0, 1)")
    lines.append("final_attention_ppe_prior    = max(d, clip(0.25*a + 0.20*p + 0.10*a*p, 0, 1))")
    lines.append("final_ppe_guardrail_policy   = max(d, clip(0.25*a + 0.50*p, 0, 1))")
    lines.append("final_learned_meta_mean      = mean(seed/model learned-meta fusion probabilities)")
    lines.append("```")
    lines.append("")
    lines.append("Recommended reporting:")
    lines.append("")
    lines.append("- Keep sequence-only risk as the clean reference danger score.")
    lines.append("- Use `final_attention_ppe_prior` when the project needs transparent, defensible attention/PPE sensitivity.")
    lines.append("- Use `final_ppe_guardrail_policy` when the demo should enforce the exp069 strong-PPE guardrail with a `0.50` bad-PPE score floor.")
    lines.append("- Use `final_safety_sensitive_rule` when the demo should aggressively warn on distracted or badly worn PPE states before zone entry.")
    lines.append("- Treat `final_learned_meta_mean` as a measured research comparator, not as the source of the safety rule.")
    lines.append("- Report threshold, hit rate, false alarms, precision, and early-warning time together.")
    path = run_dir / "final_scoring_contract.md"
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, artifact_summary, contract_df, load_df, smoke_df, readiness):
    contract_pass = bool(contract_df["pass"].all()) if not contract_df.empty else False
    lines = ["# Final Deployment Package Audit", ""]
    lines.append("This package separates what is deployment-runnable today from what is an offline research artifact.")
    lines.append("")
    lines.append("## Readiness Checklist")
    lines.append("")
    lines.append("| check | status | notes |")
    lines.append("|---|---|---|")
    for _, row in readiness.iterrows():
        lines.append(f"| {row['check']} | {row['status']} | {row['notes']} |")
    lines.append("")
    lines.append("## Artifact Groups")
    lines.append("")
    lines.append("| group | found/expected | pass | bytes | role |")
    lines.append("|---|---:|---:|---:|---|")
    for _, row in artifact_summary.iterrows():
        lines.append(f"| {row['group']} | {int(row['found_count'])}/{int(row['expected_count'])} | {row['pass']} | {int(row['bytes_total'])} | {row['role']} |")
    lines.append("")
    lines.append("## Formula Contract")
    lines.append("")
    lines.append(f"- Final score formula validation: `{'PASS' if contract_pass else 'FAIL'}`")
    if not contract_df.empty:
        worst = contract_df.sort_values("max_abs_error", ascending=False).head(5)
        lines.append("")
        lines.append("| seed | check | max abs error | pass |")
        lines.append("|---|---|---:|---:|")
        for _, row in worst.iterrows():
            lines.append(f"| {row['seed']} | {row['check']} | {float(row['max_abs_error']):.8f} | {row['pass']} |")
    lines.append("")
    lines.append("## Existing Runtime Smoke Tests")
    lines.append("")
    lines.append("| smoke | mode | rows | max risk | first alarm s | pass |")
    lines.append("|---|---|---:|---:|---:|---:|")
    for _, row in smoke_df.iterrows():
        lines.append(f"| {row['smoke']} | {row['mode']} | {int(row['prediction_rows'])} | {float(row['max_risk']):.3f} | {row['first_alarm_time_s']} | {row['pass']} |")
    lines.append("")
    lines.append("## Optimized Runtime Variants")
    lines.append("")
    lines.append("| variant | steady mean ms | steady p95 ms | steady FPS |")
    lines.append("|---|---:|---:|---:|")
    runtime_path = run_dir / "metrics" / "runtime_variant_total_summary.csv"
    runtime_df = pd.read_csv(runtime_path) if runtime_path.exists() else pd.DataFrame()
    for _, row in runtime_df.iterrows():
        lines.append(
            f"| {row['variant']} | {float(row['steady_mean_ms']):.2f} | {float(row['steady_p95_ms']):.2f} | {float(row['estimated_fps_from_steady_mean']):.2f} |"
        )
    lines.append("")
    lines.append("## Pose Resolution Runtime Tradeoff")
    lines.append("")
    lines.append("| run | variant | imgsz | steady FPS | steady p95 ms | score p95 diff vs 640 | alarm diff frames |")
    lines.append("|---|---|---:|---:|---:|---:|---:|")
    resolution_path = run_dir / "metrics" / "runtime_resolution_tradeoff.csv"
    resolution_df = pd.read_csv(resolution_path) if resolution_path.exists() else pd.DataFrame()
    for _, row in resolution_df.iterrows():
        lines.append(
            f"| {row['run_label']} | {row['variant']} | {int(row['imgsz'])} | "
            f"{float(row['estimated_fps_from_steady_mean']):.2f} | {float(row['steady_p95_ms']):.2f} | "
            f"{float(row['score_p95_abs_diff_vs_640']):.3f} | {int(row['alarm_diff_frames_vs_640'])} |"
        )
    lines.append("")
    lines.append("## Pose Backend Runtime Audit")
    lines.append("")
    lines.append("| variant | backend | hybrid FPS | hybrid mean ms | hybrid p95 ms | score p95 diff | alarm diff frames | pass mean | pass p95 |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|")
    pose_backend_path = run_dir / "metrics" / "runtime_pose_backend_audit.csv"
    pose_backend_df = pd.read_csv(pose_backend_path) if pose_backend_path.exists() else pd.DataFrame()
    for _, row in pose_backend_df.iterrows():
        lines.append(
            f"| {row['variant']} | {row['candidate_backend']} | "
            f"{float(row['estimated_hybrid_cuda_score_fps']):.2f} | "
            f"{float(row['estimated_hybrid_cuda_score_total_mean_ms']):.2f} | "
            f"{float(row['estimated_hybrid_cuda_score_total_p95_ms']):.2f} | "
            f"{float(row['score_p95_abs_diff']):.3f} | {int(row['alarm_diff_frames'])} | "
            f"{bool(row['pass_mean_30fps'])} | {bool(row['pass_p95_30fps'])} |"
        )
    lines.append("")
    lines.append("## Loadability Smoke")
    lines.append("")
    lines.append("| artifact | loadable | details |")
    lines.append("|---|---:|---|")
    for _, row in load_df.iterrows():
        lines.append(f"| {row['artifact']} | {row['loadable']} | {row['details']} |")
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- The existing `sequence_inference_demo.py` proves raw-video inference for a single TCN danger model.")
    lines.append("- The final trajectory+attention+PPE score is reproducible offline from saved same-split predictions and checked formula contracts.")
    lines.append("- `final_fused_inference_demo.py` now proves a raw-video full final-score smoke path.")
    lines.append("- `final_causal_fused_stream_smoke.py` proves a causal raw-video path with rolling attention/PPE estimates and measured latency.")
    lines.append("- `final_realtime_variant_benchmark.py` shows the core sequence-only danger model can reach about 30 FPS steady-state, while the best fused score variant remains below the strict real-time target.")
    lines.append("- `runtime_resolution_tradeoff_audit.py` shows that reducing pose input size can push fused mean FPS over 30, but p95 latency and risk-trace stability still fail the conservative gate.")
    lines.append("- `runtime_pose_backend_audit.py` shows that the available ONNX pose artifact, fixed at 320px and running on ONNX Runtime CPU here, does not close the fused real-time gap and introduces material score drift.")
    lines.append("- The remaining deployment gap is production real-time final fused operation: lightweight fused/context-stride/resolution policies need validation and optimization before claiming real-time final-score deployment.")
    lines.append("- Therefore, deployment readiness is `research-demo`: stronger than offline tables, still not a production app.")
    lines.append("")
    lines.append("## Artifacts")
    lines.append("")
    lines.append(f"- Scoring contract: `{run_dir / 'final_scoring_contract.md'}`")
    lines.append(f"- Artifact manifest: `{run_dir / 'metrics' / 'deployment_artifact_manifest.csv'}`")
    lines.append(f"- Readiness checklist: `{run_dir / 'metrics' / 'deployment_readiness_checklist.csv'}`")
    summary_path = run_dir / "final_deployment_package_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Final Deployment Package Audit", f"- Summary: `{summary_path}`")
    return summary_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    artifact_summary = artifact_group_summary(args)
    contract_df = validate_final_score_contract(args.final_score_run)
    load_df = loadability_smoke(args)
    smoke_df = inference_smoke_rows(args)
    runtime_df = runtime_variant_summary(args)
    resolution_df = runtime_resolution_summary(args)
    pose_backend_df = runtime_pose_backend_summary(args)
    readiness = readiness_rows(artifact_summary, contract_df, load_df, smoke_df, runtime_df, resolution_df, pose_backend_df)

    artifact_summary.to_csv(run_dir / "metrics" / "deployment_artifact_manifest.csv", index=False)
    contract_df.to_csv(run_dir / "metrics" / "final_score_contract_validation.csv", index=False)
    load_df.to_csv(run_dir / "metrics" / "loadability_smoke.csv", index=False)
    smoke_df.to_csv(run_dir / "metrics" / "inference_smoke_audit.csv", index=False)
    runtime_df.to_csv(run_dir / "metrics" / "runtime_variant_total_summary.csv", index=False)
    resolution_df.to_csv(run_dir / "metrics" / "runtime_resolution_tradeoff.csv", index=False)
    pose_backend_df.to_csv(run_dir / "metrics" / "runtime_pose_backend_audit.csv", index=False)
    readiness.to_csv(run_dir / "metrics" / "deployment_readiness_checklist.csv", index=False)
    write_json(
        run_dir / "metrics" / "deployment_readiness_summary.json",
        {
            "artifact_groups_pass": bool(artifact_summary["pass"].all()),
            "score_contract_pass": bool(contract_df["pass"].all()) if not contract_df.empty else False,
            "sample_loadability_pass": bool(load_df["loadable"].all()) if not load_df.empty else False,
            "single_sequence_smoke_pass": bool(
                len(smoke_df[smoke_df["smoke"].isin(["precomputed_pose_single_tcn", "raw_video_single_tcn"])]) >= 2
                and smoke_df[smoke_df["smoke"].isin(["precomputed_pose_single_tcn", "raw_video_single_tcn"])]["pass"].all()
            )
            if not smoke_df.empty
            else False,
            "full_final_fused_raw_video_runtime": "pass" if bool(smoke_df[smoke_df["smoke"].eq("raw_video_final_fused")]["pass"].all()) else "gap",
            "causal_final_fused_raw_video_runtime": "pass" if bool(smoke_df[smoke_df["smoke"].eq("causal_raw_video_final_fused")]["pass"].all()) else "gap",
            "optimized_runtime_variant_benchmark": "pass" if not runtime_df.empty else "gap",
            "pose_resolution_runtime_tradeoff": "pass" if not resolution_df.empty else "gap",
            "pose_backend_runtime_audit": "pass" if not pose_backend_df.empty else "gap",
            "core_sequence_realtime_candidate": "pass" if bool(readiness[readiness["check"].eq("core_sequence_realtime_candidate")]["status"].eq("pass").all()) else "gap",
            "production_realtime_final_fused_runtime": "pass" if bool(readiness[readiness["check"].eq("production_realtime_final_fused_runtime")]["status"].eq("pass").all()) else "gap",
        },
    )
    write_scoring_contract(run_dir)
    summary_path = write_summary(run_dir, artifact_summary, contract_df, load_df, smoke_df, readiness)
    print(run_dir)
    print(summary_path)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Audit deployment readiness for final machine-safety scoring artifacts.")
    parser.add_argument("--run-name", default="exp_048_final_deployment_package")
    parser.add_argument("--fusion-run", default="runs/exp_020_same_split_fusion")
    parser.add_argument("--final-score-run", default="runs/exp_039_final_aggregated_score")
    parser.add_argument("--final-subclip-run", default="runs/exp_047_final_aggregated_subclip_eval")
    parser.add_argument("--sequence-model", default="runs/exp_010_sequence_len60_focal_catalogue/models/tcn_aug.pt")
    parser.add_argument("--sequence-features", default="runs/exp_010_sequence_len60_focal_catalogue/features/sequence_dataset.npz")
    parser.add_argument("--precomputed-smoke", default="runs/exp_018_inference_smoke_tcn_aug")
    parser.add_argument("--raw-smoke", default="runs/exp_019_raw_video_inference_smoke_tcn_aug")
    parser.add_argument("--final-fused-smoke", default="runs/exp_049_final_fused_raw_video_inference_smoke")
    parser.add_argument("--causal-fused-smoke", default="runs/exp_051_causal_final_fused_stream_smoke")
    parser.add_argument("--runtime-variant-run", default="runs/exp_053_realtime_variant_benchmark")
    parser.add_argument("--resolution-tradeoff-run", default="runs/exp_059_runtime_resolution_tradeoff")
    parser.add_argument("--pose-backend-run", default="runs/exp_064_runtime_pose_backend_audit")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_048_final_deployment_package_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["final_deployment_package_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
